# IOAI — 2025 Stage 1 Ecg Anomaly Detection (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/OlimpiadaAI/II-OlimpiadaAI"
CLONE = "II-OlimpiadaAI"
SUBDIR = "1_etap/3_wykrywanie_zaburzen_sygnalu_ekg"
WORKDIR = "1_etap/3_wykrywanie_zaburzen_sygnalu_ekg"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Wykrywanie Zaburzeń Sygnału EKG — EKG 시계열 5분류 (Polish AI Olympiad II · 1etap)

단일유도 **EKG 시계열**(길이 150)을 5개 클래스로 분류한다: `normal / afib / pac / pvc / st_elevation`(라벨 0~4).

- `train_validation_sets.npz` — `X_train`(2000,150)·`y_train`(2000), `X_validation`(1500,150)·`y_validation`(1500)
- **제출**: `X_validation` 각 표본에 대한 예측을 `submission.csv`(`id,label`, id=0..1499)로 저장 → accuracy 채점

아래는 간단한 베이스라인(특징 정규화 + 분류기)입니다. 원본 폴란드어 문제·모범답안은 Solution 탭 참고.

## 데이터

In [ ]:
import numpy as np, pandas as pd
d = np.load("train_validation_sets.npz")
Xtr, ytr = d["X_train"].astype(np.float32), d["y_train"].astype(int)
Xva = d["X_validation"].astype(np.float32)
print(Xtr.shape, Xva.shape, "classes:", sorted(set(ytr)))

## 베이스라인 — 통계특징 + HistGBM

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
# 간단한 시계열 통계특징 + 원신호
def feats(X):
    stat = np.stack([X.mean(1), X.std(1), X.min(1), X.max(1),
                     np.diff(X,axis=1).std(1), (X>X.mean(1,keepdims=True)).mean(1)], axis=1)
    return np.concatenate([X, stat], axis=1)
sc = StandardScaler().fit(feats(Xtr))
clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=0)
clf.fit(sc.transform(feats(Xtr)), ytr)

## 예측 → submission.csv

In [ ]:
pred = clf.predict(sc.transform(feats(Xva)))
pd.DataFrame({"id": np.arange(len(pred)), "label": pred.astype(int)}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred))

더 끌어올리려면 1D-CNN(원신호)·RR-간격 등 EKG 도메인 특징·앙상블을 시도하세요.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)